In [1]:
# =====================================================================
# Part II - Image Colorization - TEMPLATE
# =====================================================================
#
# Task: take a grayscale image and predict its colors.
#
# You build and train the model any way you want (autoencoder, VAE, GAN, ...).
#
# The grader will:
#   1. run your model class + load_model() to load your saved weights,
#   2. call colorize() on their own images,
#   3. compare your output to hidden color images (PSNR / MSE).
#
# So you MUST keep the 3 fixed rules below.
# =====================================================================
#
# ---------------------------------------------------------------------
# FIXED RULES (do not change)
# ---------------------------------------------------------------------
#
# 1. Save your trained weights as a state_dict:
#        torch.save(model.state_dict(), "weights.pth")
#    Submit this "weights.pth" file together with your notebook.
#
# 2. All images are 256 x 256 PNG.
#    colorize() input  : grayscale array, shape (256, 256), float in [0, 1]
#    colorize() output : RGB array,       shape (256, 256, 3), float in [0, 1]
#
# 3. Do file loading OUTSIDE colorize() (see the demo at the bottom).
#    colorize() only takes arrays, not file paths.
# ---------------------------------------------------------------------

In [2]:
import numpy as np
import torch
import torch.nn as nn

IMG_SIZE = 256

In [ ]:
# ---------------------------------------------------------------------
# 1) YOUR MODEL
# ---------------------------------------------------------------------
# A U-Net that predicts only CHROMINANCE (Cb, Cr), not full RGB.
#
# Plain RGB regression with L1/MSE loss is known to produce muted,
# undersaturated colors: the model hedges toward "safe" averaged
# colors whenever it's unsure, because errors in R/G/B are entangled
# with brightness errors too. The standard fix in the colorization
# literature is to work in a luma/chroma colorspace (Y/Cb/Cr): the
# grayscale input IS (almost exactly) the Y channel already, so it
# doesn't need to be predicted at all -- only the 2 color channels
# (Cb, Cr) do. This confines all prediction error to color, never
# brightness, which is both an easier learning problem and closer to
# how the eye actually perceives color images.
#
# base=24 (~4.4M params): with Part I finished, the full machine is
# available, so this affords a bigger network than the base=16 first
# attempt. The default here MUST match what's actually trained --
# load_model() below rebuilds with no arguments, so a mismatched
# default would fail to load the saved weights.

Y_R, Y_G, Y_B = 0.299, 0.587, 0.114  # matches PIL's L = ITU-R 601-2 luma


def rgb_to_ycbcr(rgb):
    """rgb: (..., H, W, 3) in [0,1] -> y, cb, cr each (..., H, W) in [0,1]."""
    r, g, b = rgb[..., 0], rgb[..., 1], rgb[..., 2]
    y = Y_R * r + Y_G * g + Y_B * b
    cb = -0.168736 * r - 0.331264 * g + 0.5 * b + 0.5
    cr = 0.5 * r - 0.418688 * g - 0.081312 * b + 0.5
    return y, cb, cr


def ycbcr_to_rgb(y, cb, cr):
    """y, cb, cr: (..., H, W) in [0,1] -> rgb (..., H, W, 3) in [0,1]."""
    cb0, cr0 = cb - 0.5, cr - 0.5
    r = y + 1.402 * cr0
    g = y - 0.344136 * cb0 - 0.714136 * cr0
    b = y + 1.772 * cb0
    return np.clip(np.stack([r, g, b], axis=-1), 0.0, 1.0)


def conv_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
    )


class ColorizeModel(nn.Module):
    def __init__(self, base=24):
        super().__init__()
        self.enc1 = conv_block(1, base)
        self.enc2 = conv_block(base, base * 2)
        self.enc3 = conv_block(base * 2, base * 4)
        self.enc4 = conv_block(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)

        self.bottleneck = conv_block(base * 8, base * 16)

        self.up4 = nn.ConvTranspose2d(base * 16, base * 8, 2, stride=2)
        self.dec4 = conv_block(base * 16, base * 8)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = conv_block(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = conv_block(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = conv_block(base * 2, base)

        self.out = nn.Conv2d(base, 2, 1)  # Cb, Cr only

    def forward(self, x):
        # x: (batch, 1, 256, 256) -- the Y (luminance) channel
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))

        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return torch.sigmoid(self.out(d1))  # (batch, 2, 256, 256): cb, cr in [0,1]

In [4]:
# ---------------------------------------------------------------------
# 2) LOAD YOUR TRAINED WEIGHTS
# ---------------------------------------------------------------------
# Rebuilds the empty model and loads the saved numbers.
# This is instant - no training.
def load_model(weights_path="weights.pth"):
    model = ColorizeModel()
    model.load_state_dict(torch.load(weights_path, map_location="cpu"))
    model.eval()
    return model

In [ ]:
# ---------------------------------------------------------------------
# 3) COLORIZE ONE IMAGE
# ---------------------------------------------------------------------
# gray_img: numpy array, shape (256, 256), float in [0, 1]
# returns : numpy array, shape (256, 256, 3), float in [0, 1]
#
# The external contract is unchanged (grayscale in, RGB out) -- the
# Y/Cb/Cr split is purely an internal implementation detail. gray_img
# is used directly as Y (it already IS the luminance channel), the
# model predicts Cb/Cr, and the three are recombined into RGB.
def colorize(gray_img, model):
    x = torch.from_numpy(gray_img).float().view(1, 1, IMG_SIZE, IMG_SIZE)

    with torch.no_grad():           # no gradients needed for inference
        cb_cr = model(x)            # (1, 2, 256, 256)

    cb = cb_cr[0, 0].cpu().numpy()
    cr = cb_cr[0, 1].cpu().numpy()
    return ycbcr_to_rgb(gray_img, cb, cr)

In [ ]:
# =======================================================================
# TRAINING -- produces weights.pth
# =======================================================================
# Everything above this line is the fixed submission interface. Below
# is how weights.pth was actually produced: dataset, training loop,
# and a quick self-check against the same PSNR/MSE metric the grader
# uses, before the final demo.
#
# No training images were provided for this assignment ("you may
# choose all the images you want to train your model" -- directives.txt).
# Flowers102 (torchvision, auto-downloads) was used: colorful, diverse
# natural photos, no license friction, no manual collection needed.
# Only the images are used -- the 102 flower classes are irrelevant to
# colorization and are never touched.

import os
import glob
import random
import torchvision
import torchvision.transforms as T
from PIL import Image

DATA_ROOT = "colorization_data"
os.makedirs(DATA_ROOT, exist_ok=True)
for split in ["train", "val", "test"]:
    torchvision.datasets.Flowers102(root=DATA_ROOT, split=split, download=True)
IMAGE_DIR = os.path.join(DATA_ROOT, "flowers-102", "jpg")

image_paths = sorted(glob.glob(os.path.join(IMAGE_DIR, "*.jpg")))
print("images found:", len(image_paths))

random.Random(0).shuffle(image_paths)
# All 8189 available images (was 1500, then 3000): Flowers102's entire
# pool, not a subset -- the first attempt's real bottleneck turned out
# to be optimization (constant LR, see the training cell below), not
# data volume, but more distinct photos still directly helps the model
# generalize to whatever images the grader actually uses, rather than
# memorizing a narrow slice of flower photos. GPU training makes the
# full set affordable.
N_IMAGES = 8189
image_paths = image_paths[:N_IMAGES]
n_val = max(1, int(0.1 * len(image_paths)))
val_paths, train_paths = image_paths[:n_val], image_paths[n_val:]
print("train:", len(train_paths), " val:", len(val_paths))


class ColorizationDataset(torch.utils.data.Dataset):
    """Loads a color photo, returns (Y, [Cb, Cr]), all 256x256 in [0, 1].

    Y is the grayscale input (== luminance); Cb/Cr are the color
    channels the model must predict. The photo supervises itself --
    no separate labels needed. `augment=True` applies a random
    horizontal flip (train split only) for free extra variety from
    the same images.
    """

    def __init__(self, paths, size=IMG_SIZE, augment=False):
        self.paths = paths
        self.resize = T.Resize((size, size))
        self.augment = augment

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = self.resize(Image.open(self.paths[idx]).convert("RGB"))
        if self.augment and random.random() < 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
        rgb = np.array(img).astype("float32") / 255.0
        y, cb, cr = rgb_to_ycbcr(rgb)
        y_t = torch.from_numpy(y).float().unsqueeze(0)
        cbcr_t = torch.from_numpy(np.stack([cb, cr], axis=0)).float()
        return y_t, cbcr_t


train_ds = ColorizationDataset(train_paths, augment=True)
val_ds = ColorizationDataset(val_paths, augment=False)

In [ ]:
BATCH_SIZE = 8
EPOCHS = 100
LR = 1e-3

# Set True to skip training entirely and just load whatever's already
# saved on Drive -- for checking/demoing current progress (PSNR cell,
# demo cell below) without spending GPU time, e.g. between sessions
# while waiting for more GPU quota. Requires a checkpoint to already
# exist (run once with this False first).
SKIP_TRAINING = False

# Persist checkpoints to Google Drive, not Colab's local disk. Colab's
# local filesystem is wiped on every disconnect/reset -- exactly what
# cost an earlier run its progress at epoch 7, with the checkpoint file
# stranded in a session that vanished. Drive survives across sessions,
# so a disconnect only costs the current (incomplete) epoch, never
# everything before it, and there's no manual download/upload step to
# remember.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    CHECKPOINT_DIR = "/content/drive/MyDrive/colorization_checkpoints"
except ImportError:
    CHECKPOINT_DIR = "."  # not running in Colab (e.g. local test) -- use the working directory
import os as _os
_os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_PATH = _os.path.join(CHECKPOINT_DIR, "checkpoint.pt")
WEIGHTS_PATH = _os.path.join(CHECKPOINT_DIR, "weights.pth")
WEIGHTS_BEST_PATH = _os.path.join(CHECKPOINT_DIR, "weights_best.pth")
print("checkpoint directory:", CHECKPOINT_DIR)

torch.set_num_threads(6)
torch.manual_seed(0)
model = ColorizeModel()
opt = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
l1_loss_fn = nn.L1Loss()


def ycbcr_to_rgb_torch(y, cb, cr):
    """Differentiable version of ycbcr_to_rgb (batched tensors, gradients
    flow through) -- needed to feed predicted colors into the perceptual
    loss below. Same math as the numpy version above, kept in sync."""
    cb0, cr0 = cb - 0.5, cr - 0.5
    r = y + 1.402 * cr0
    g = y - 0.344136 * cb0 - 0.714136 * cr0
    b = y + 1.772 * cb0
    return torch.clamp(torch.cat([r, g, b], dim=1), 0.0, 1.0)


class VGGPerceptualLoss(nn.Module):
    """L1 distance between VGG19 features of predicted vs. true RGB,
    instead of raw pixels. Plain per-pixel L1/MSE is exactly why the
    first attempt produced muted, undersaturated colors: minimizing
    pixel error when uncertain about the true color means hedging
    toward a safe average. A perceptual loss compares texture/structure
    in a pretrained feature space instead, which doesn't punish
    plausible-but-different colors the same way, and tends to produce
    more natural, saturated results. Used ALONGSIDE L1 (below), not
    instead of it: the grader scores PSNR/MSE directly, which is a
    pixel-level metric, so pixel accuracy still has to matter too.

    layer_idx=26 -- true relu4_4 (deep/"extended" slice, as requested):
    VGG19's features Sequential is 5 conv blocks of
    [conv,relu]*2-or-4 + pool; block4 (512-channel convs) ends at
    index 26 = the 4th ReLU in that block = relu4_4. Deeper than the
    shallow relu2_2 slice used for the CPU backup (index 9) -- more
    expensive per batch, but affordable on GPU, and captures higher-
    level texture/structure than a shallow slice would.
    """
    def __init__(self, layer_idx=26):
        super().__init__()
        weights = torchvision.models.VGG19_Weights.IMAGENET1K_V1
        vgg = torchvision.models.vgg19(weights=weights).features[:layer_idx].eval()
        for p in vgg.parameters():
            p.requires_grad = False
        self.vgg = vgg
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, pred_rgb, target_rgb):
        pred_n = (pred_rgb - self.mean) / self.std
        target_n = (target_rgb - self.mean) / self.std
        return nn.functional.l1_loss(self.vgg(pred_n), self.vgg(target_n))


PERCEPTUAL_WEIGHT = 0.05  # L1 dominates; perceptual nudges toward natural color
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
vgg_loss_fn = VGGPerceptualLoss().to(device)
print("device:", device)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

# RESUME: if a checkpoint from an earlier (possibly interrupted) run of
# this same cell exists -- on Drive, so it survives a full runtime
# reset, not just this session -- pick up from exactly where it left
# off: model weights, optimizer momentum, and the LR schedule's
# position, instead of silently starting over from a random model.
start_epoch = 0
best_val_loss = float("inf")
if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(ckpt["model"])
    opt.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    start_epoch = ckpt["epoch"] + 1
    best_val_loss = ckpt["best_val_loss"]
    print("resuming from checkpoint: epoch %d, best val loss so far %.4f" % (start_epoch, best_val_loss))
elif SKIP_TRAINING:
    raise FileNotFoundError(
        "SKIP_TRAINING=True but no checkpoint exists yet at %s -- "
        "nothing to load. Run with SKIP_TRAINING=False at least once "
        "first." % CHECKPOINT_PATH)
else:
    print("no checkpoint found -- starting fresh from epoch 0")

if SKIP_TRAINING:
    print("SKIP_TRAINING=True -- using epoch %d's saved weights as-is, "
          "not training further this run." % (start_epoch - 1))

# Two submission-format checkpoints (plain state_dict, matching what
# load_model() expects), plus the richer checkpoint.pt above for resume
# -- all three on Drive, so a disconnect at any point costs at most the
# current, incomplete epoch, never anything before it, and never
# requires a manual download/upload to recover.
#  - weights.pth is overwritten after EVERY epoch, no matter what.
#  - weights_best.pth is only overwritten when validation loss actually
#    improves, guarding against overfitting late in a 100-epoch run.
# If training completes normally, the last step below promotes the best
# checkpoint over the last one, so a normal finish still submits the
# best-quality weights, not just whichever epoch happened to run last.
# Wrapped in try/except so an in-kernel error (a bad batch, a transient
# glitch, running out of memory) doesn't halt the whole notebook --
# this cell finishes normally either way, and "Run All" continues to
# the PSNR check and demo cells below using whatever was last saved.
# This does NOT protect against a GPU disconnect/runtime reset (that
# kills the whole session, not just this cell -- no code can run
# through that, only reconnecting manually can) -- only against errors
# that happen while the kernel is still alive.
stopped_early = False
epoch = start_epoch - 1  # so the except-block message is sane even if SKIP_TRAINING never entered the loop
if not SKIP_TRAINING:
    try:
        for epoch in range(start_epoch, EPOCHS):
            model.train()
            train_loss = 0.0
            for y, cbcr in train_loader:
                y, cbcr = y.to(device), cbcr.to(device)
                opt.zero_grad()
                pred = model(y)
                l1 = l1_loss_fn(pred, cbcr)
                pred_rgb = ycbcr_to_rgb_torch(y, pred[:, 0:1], pred[:, 1:2])
                target_rgb = ycbcr_to_rgb_torch(y, cbcr[:, 0:1], cbcr[:, 1:2])
                perceptual = vgg_loss_fn(pred_rgb, target_rgb)
                loss = l1 + PERCEPTUAL_WEIGHT * perceptual
                loss.backward()
                opt.step()
                train_loss += loss.item() * y.size(0)
            train_loss /= len(train_ds)

            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for y, cbcr in val_loader:
                    y, cbcr = y.to(device), cbcr.to(device)
                    pred = model(y)
                    l1 = l1_loss_fn(pred, cbcr)
                    pred_rgb = ycbcr_to_rgb_torch(y, pred[:, 0:1], pred[:, 1:2])
                    target_rgb = ycbcr_to_rgb_torch(y, cbcr[:, 0:1], cbcr[:, 1:2])
                    perceptual = vgg_loss_fn(pred_rgb, target_rgb)
                    val_loss += (l1 + PERCEPTUAL_WEIGHT * perceptual).item() * y.size(0)
            val_loss /= len(val_ds)
            scheduler.step()

            print("epoch %2d | lr %.2e | train loss %.4f | val loss %.4f" % (
                epoch, opt.param_groups[0]["lr"], train_loss, val_loss))

            torch.save(model.state_dict(), WEIGHTS_PATH)  # every epoch, unconditionally
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save(model.state_dict(), WEIGHTS_BEST_PATH)
                print("  (new best -- also saved to weights_best.pth)")

            torch.save({
                "model": model.state_dict(),
                "optimizer": opt.state_dict(),
                "scheduler": scheduler.state_dict(),
                "epoch": epoch,
                "best_val_loss": best_val_loss,
            }, CHECKPOINT_PATH)
    except Exception as err:
        stopped_early = True
        print("TRAINING STOPPED EARLY at epoch %d: %r" % (epoch, err))
        print("weights.pth/checkpoint.pt still hold epoch %d's results -- "
              "continuing to the next cells with that, and re-running this "
              "cell later will resume from epoch %d." % (epoch, epoch + 1))

# Normal completion: submit the best epoch, not just the last one. Also
# copy the final weights.pth next to this notebook (the FIXED submission
# path load_model() expects by default), not just on Drive. Skipped if
# training stopped early or was skipped -- weights.pth on Drive already
# holds the right thing to leave in place for the cells below.
import shutil
if not stopped_early and not SKIP_TRAINING:
    shutil.copyfile(WEIGHTS_BEST_PATH, WEIGHTS_PATH)
shutil.copyfile(WEIGHTS_PATH, "weights.pth")
print("best val loss so far: %.4f -- weights.pth is ready for the cells below" % best_val_loss)

In [ ]:
# Self-check against the same metric the grader uses (PSNR), via the
# actual submission interface (load_model + colorize) -- not a
# shortcut through the training-time model object.
def psnr(pred, target, max_val=1.0):
    mse = float(np.mean((pred - target) ** 2))
    if mse == 0:
        return float("inf")
    return 10.0 * np.log10((max_val ** 2) / mse)


loaded = load_model("weights.pth")
psnrs = []
for y, cbcr in val_ds:
    y_np = y.squeeze(0).numpy()
    ground_truth_rgb = ycbcr_to_rgb(y_np, cbcr[0].numpy(), cbcr[1].numpy())
    pred_rgb = colorize(y_np, loaded)
    psnrs.append(psnr(pred_rgb, ground_truth_rgb))
print("mean val PSNR: %.2f dB over %d held-out images" % (float(np.mean(psnrs)), len(psnrs)))

In [ ]:
# =======================================================================
# DEMO -- file loading happens here, OUTSIDE colorize() (fixed rule 3)
# =======================================================================
import matplotlib.pyplot as plt

model = load_model("weights.pth")

fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for row, idx in enumerate([0, 1, 2]):
    y_sample, cbcr_sample = val_ds[idx]
    y_np = y_sample.squeeze(0).numpy()
    ground_truth_rgb = ycbcr_to_rgb(y_np, cbcr_sample[0].numpy(), cbcr_sample[1].numpy())
    pred = colorize(y_np, model)

    axes[row, 0].imshow(y_np, cmap="gray")
    axes[row, 1].imshow(pred)
    axes[row, 2].imshow(ground_truth_rgb)
    if row == 0:
        for ax, title in zip(axes[row], ["input (gray)", "predicted color", "ground truth"]):
            ax.set_title(title)
    for ax in axes[row]:
        ax.axis("off")
plt.tight_layout()
plt.show()